# Validating Adaptive Query Execution (AQE) in Databricks

AQE re-optimizes a query **at runtime** using real statistics instead of the
optimizer's pre-execution estimates. Its three benefits only become visible on a
**shuffle-heavy join over skewed data** — which is exactly what this notebook builds.

| AQE feature | Config key | What it does |
|---|---|---|
| Master switch | `spark.sql.adaptive.enabled` | Turns AQE on/off (default **true** in Databricks) |
| Coalesce shuffle partitions | `spark.sql.adaptive.coalescePartitions.enabled` | Merges many tiny post-shuffle partitions into fewer |
| Skew-join handling | `spark.sql.adaptive.skewJoin.enabled` | Splits an oversized (skewed) partition so one task isn't a straggler |
| Target partition size | `spark.sql.adaptive.advisoryPartitionSizeInBytes` | Advisory size used when coalescing (default 64MB) |

**The point of the test:** we force a sort-merge join on data where ~85% of the rows
share a single join key. With AQE off, one task handles that giant partition and becomes
a straggler. With AQE on, the skewed partition is split and the job rebalances.

## 1. Parameters
Tune `FACT_ROWS` to your cluster. 60M shows a clear difference on a small cluster;
drop to 10–20M on a single-node cluster, raise to 200M+ on a larger one.
AQE's benefit grows with scale and skew, so very small data may show little difference.

In [1]:
FACT_ROWS    = 60_000_000   # rows in the large (fact) table
DISTINCT_KEY = 1_000_000    # number of distinct customer ids in the dimension
HOT_KEY_FRAC = 0.85         # fraction of fact rows that collapse onto a single hot key (the skew)
DB           = "aqe_demo"   # schema used for the temp tables

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
print(f"fact rows={FACT_ROWS:,} | distinct keys={DISTINCT_KEY:,} | hot-key fraction={HOT_KEY_FRAC}")

NameError: name 'spark' is not defined

## 2. Generate a deliberately skewed dataset
`fact.cust_id` is the hot key (≈85% of rows are id 0); `dim` is a uniform lookup table.
We persist both to Delta **once** so the benchmark measures the join/shuffle, not data generation.

In [ ]:
from pyspark.sql import functions as F

# Large fact table with a skewed join key
fact = (
    spark.range(FACT_ROWS)
    .withColumn(
        "cust_id",
        F.when(F.rand(seed=11) < HOT_KEY_FRAC, F.lit(0))                 # hot key 0
         .otherwise((F.rand(seed=12) * DISTINCT_KEY).cast("long"))       # uniform tail
    )
    .withColumn("amount", (F.rand(seed=13) * 100).cast("double"))
)

# Dimension table — uniform, one row per customer id (includes id 0)
dim = (
    spark.range(DISTINCT_KEY)
    .withColumnRenamed("id", "cust_id")
    .withColumn("segment", (F.rand(seed=14) * 5).cast("int"))
    .withColumn("name", F.concat(F.lit("cust_"), F.col("cust_id")))
)

(fact.write.mode("overwrite").format("delta").saveAsTable(f"{DB}.fact"))
(dim.write.mode("overwrite").format("delta").saveAsTable(f"{DB}.dim"))

print("Tables written.")

### Confirm the skew is real
One key should dominate the distribution — that imbalance is what AQE fixes.

In [ ]:
(spark.table(f"{DB}.fact")
      .groupBy("cust_id").count()
      .orderBy(F.desc("count"))
      .show(5, truncate=False))

## 3. Benchmark harness
We write to the `noop` sink — it executes the full plan (shuffle + join + aggregate) but
discards output, so we time the computation rather than result collection or a real write.

In [ ]:
import time

def run_join():
    f = spark.table(f"{DB}.fact")
    d = spark.table(f"{DB}.dim")
    return (
        f.join(d, "cust_id")                       # shuffle join on the skewed key
         .groupBy("segment")
         .agg(F.sum("amount").alias("total"))
    )

def benchmark(label):
    start = time.time()
    run_join().write.format("noop").mode("overwrite").save()
    elapsed = time.time() - start
    print(f"{label:<28} {elapsed:8.1f} s")
    return elapsed

## 4. Baseline — AQE OFF
We disable AQE and also disable auto-broadcast (`autoBroadcastJoinThreshold = -1`) so Spark
is forced into a **sort-merge join**, where the skewed partition actually hurts.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.shuffle.partitions", 200)

t_off = benchmark("AQE OFF (baseline)")

# Inspect the plan: expect a plain SortMergeJoin, no AdaptiveSparkPlan node
run_join().explain(mode="simple")

## 5. AQE ON
Same query, broadcast still disabled — so the only thing that changed is AQE.
The skew-join optimizer should split the hot partition and the runtime should drop.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # keep broadcast off: isolate skew handling

t_on = benchmark("AQE ON")

# Plan now wraps in an AdaptiveSparkPlan node; after execution the final plan shows the skew split
run_join().explain(mode="simple")

## 6. Result

In [ ]:
speedup = t_off / t_on if t_on else float("nan")
print("=" * 44)
print(f"  AQE OFF : {t_off:8.1f} s")
print(f"  AQE ON  : {t_on:8.1f} s")
print(f"  Speedup : {speedup:8.2f}x")
print("=" * 44)
print("\nThe gain comes from the skewed-partition split: with AQE off, one task")
print("processes the ~85% hot key and straggles; with AQE on it is divided across tasks.")

## 7. What to look at in the Spark UI (the qualitative proof)
Open the **Spark UI → SQL / DataFrame** tab and click into each run:

- **AQE OFF run:** the sort-merge join stage has one task whose *shuffle read* and
  *duration* dwarf the others — the straggler on the hot key.
- **AQE ON run:** the join node is labelled **`skew=true`** and the skewed partition is
  split into several tasks; you'll also see an **`AQEShuffleRead`** node where partitions
  were coalesced. Task durations are far more even.

That `skew=true` label and the balanced task times are the visual evidence to screenshot.

## 8. (Optional) See AQE switch join strategy at runtime
With auto-broadcast **enabled**, AQE can convert a sort-merge join to a broadcast join once it
measures that one side is small at runtime — visible as `BroadcastHashJoin` in the final plan.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10MB

# Filter the dimension down so it becomes broadcast-eligible only at runtime
small = spark.table(f"{DB}.dim").filter("segment = 1")
joined = spark.table(f"{DB}.fact").join(small, "cust_id")
joined.write.format("noop").mode("overwrite").save()
joined.explain(mode="simple")   # look for BroadcastHashJoin in the adaptive plan

## 9. Cleanup

In [ ]:
# Reset configs to Databricks defaults
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.shuffle.partitions")

# Drop the demo tables (uncomment to remove)
# spark.sql(f"DROP SCHEMA IF EXISTS {DB} CASCADE")
print("Done.")